<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/27_rag_error_analysis/rag_error_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas numpy sentence-transformers scikit-learn

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [38]:
documents = [
    "Elon Musk founded SpaceX in 2002.",
    "SpaceX is a private aerospace company focused on space exploration.",
    "Elon Musk is also the CEO of Tesla.",
    "Tesla is an electric vehicle company.",
    "Tesla is headquartered in Austin, Texas."
]

df_docs = pd.DataFrame({"text": documents})
df_docs

,text
0,Elon Musk founded SpaceX in 2002.
1,SpaceX is a private aerospace company focused ...
2,Elon Musk is also the CEO of Tesla.
3,Tesla is an electric vehicle company.
4,"Tesla is headquartered in Austin, Texas."


In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedding_model.encode(df_docs['text'].tolist())

In [ ]:
def retrieve(query, top_k=2):
    query_embedding = embedding_model.encode([query])
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    top_indices = np.argsort(scores)[-top_k:][::-1]
    return df_docs.iloc[top_indices]

In [ ]:
def generate_followup_query(query):
    if "elon musk" in query.lower():
        return "Which company of Elon Musk is related to space?"
    return query

In [ ]:
def clean_context(context_1, context_2):
    sentences = (context_1 + " " + context_2).split(".")
    unique = []
    for s in sentences:
        s = s.strip()
        if s and s not in unique:
            unique.append(s)
    return ". ".join(unique)

In [ ]:
from sentence_transformers import util

answers = ["Elon Musk", "SpaceX", "Tesla", "Austin"]
answer_embeddings = embedding_model.encode(answers)

def extract_answer(context, query):
    query_embedding = embedding_model.encode(query)

    scores = util.cos_sim(query_embedding, answer_embeddings)[0]
    best_idx = scores.argmax()

    return answers[best_idx]

In [ ]:
def multi_hop_rag(query):
    # Step 1: First Retrieval
    docs_1 = retrieve(query)
    context_1 = " ".join(docs_1['text'].tolist())

    # Step 2: Follow-up Query
    followup_query = generate_followup_query(query)

    # Step 3: Second Retrieval
    docs_2 = retrieve(followup_query)
    context_2 = " ".join(docs_2['text'].tolist())

    # Step 4: Clean Context
    final_context = clean_context(context_1, context_2)

    # Step 5: Answer Extraction
    answer = extract_answer(final_context, query)

    return answer

In [39]:
data = [
    {"query": "Who started SpaceX company?", "true": "Elon Musk"},
    {"query": "Which company is involved in space exploration?", "true": "SpaceX"},
    {"query": "Name a company owned by Elon Musk related to cars", "true": "Tesla"},
    {"query": "Where is Tesla located?", "true": "Austin"},
    {"query": "Which company works in rockets?", "true": "SpaceX"}
]

df = pd.DataFrame(data)
df

,query,true
0,Who started SpaceX company?,Elon Musk
1,Which company is involved in space exploration?,SpaceX
2,Name a company owned by Elon Musk related to cars,Tesla
3,Where is Tesla located?,Austin
4,Which company works in rockets?,SpaceX


In [40]:
df["predicted"] = df["query"].apply(multi_hop_rag)
df

,query,true,predicted
0,Who started SpaceX company?,Elon Musk,SpaceX
1,Which company is involved in space exploration?,SpaceX,SpaceX
2,Name a company owned by Elon Musk related to cars,Tesla,Elon Musk
3,Where is Tesla located?,Austin,Tesla
4,Which company works in rockets?,SpaceX,SpaceX


In [41]:
df["correct"] = (df["true"].str.lower() == df["predicted"].str.lower())
df

,query,true,predicted,correct
0,Who started SpaceX company?,Elon Musk,SpaceX,False
1,Which company is involved in space exploration?,SpaceX,SpaceX,True
2,Name a company owned by Elon Musk related to cars,Tesla,Elon Musk,False
3,Where is Tesla located?,Austin,Tesla,False
4,Which company works in rockets?,SpaceX,SpaceX,True


In [42]:
errors = df[df["correct"] == False]
errors

,query,true,predicted,correct
0,Who started SpaceX company?,Elon Musk,SpaceX,False
2,Name a company owned by Elon Musk related to cars,Tesla,Elon Musk,False
3,Where is Tesla located?,Austin,Tesla,False


In [43]:
def categorize_error(row):
    if row["predicted"] == "Answer not found":
        return "False Negative"
    else:
        return "Wrong Answer"

errors["error_type"] = errors.apply(categorize_error, axis=1)
errors

/tmp/ipykernel_5314/1656593361.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors["error_type"] = errors.apply(categorize_error, axis=1)


,query,true,predicted,correct,error_type
0,Who started SpaceX company?,Elon Musk,SpaceX,False,Wrong Answer
2,Name a company owned by Elon Musk related to cars,Tesla,Elon Musk,False,Wrong Answer
3,Where is Tesla located?,Austin,Tesla,False,Wrong Answer


In [44]:
print("🔍 Total Queries:", len(df))
print("❌ Total Errors:", len(errors))

print("\n📊 Error Breakdown:")
print(errors["error_type"].value_counts())

🔍 Total Queries: 5
❌ Total Errors: 3

📊 Error Breakdown:
error_type
Wrong Answer    3
Name: count, dtype: int64


In [45]:
accuracy = df["correct"].mean()
print("\n✅ Accuracy:", accuracy)


✅ Accuracy: 0.4


In [46]:
for i, row in errors.iterrows():
    print("\n----------------------------")
    print("Query:", row["query"])
    print("Expected:", row["true"])
    print("Predicted:", row["predicted"])
    print("Error Type:", row["error_type"])


----------------------------
Query: Who started SpaceX company?
Expected: Elon Musk
Predicted: SpaceX
Error Type: Wrong Answer

----------------------------
Query: Name a company owned by Elon Musk related to cars
Expected: Tesla
Predicted: Elon Musk
Error Type: Wrong Answer

----------------------------
Query: Where is Tesla located?
Expected: Austin
Predicted: Tesla
Error Type: Wrong Answer
